**Repository copy — code and narrative only.** Outputs and embedded data have been removed. Numerical findings in the narrative describe the original October 2026 run, not a new execution of this copy. See `../../README.md` for data placement and rebuild order. Original local notebooks are unchanged.

# Eligibility audit and vocabulary change, 2018–2019 versus 2022–2023
## STAT 236 · School board meetings

**Question:** Is the average within-district vocabulary composition different between the two periods? Compare Higher Criticism (HC+), the minimum-p/max-|Z| test, and a dense sum-of-squares test using the same observations and calibration.

This notebook contains the completed screening, matched-panel analysis, global tests, sensitivity checks, and a conditional Gaussian power illustration. It uses **every previously cleaned transcript passing the stated eligibility and panel rules**, rather than a random transcript sample. It does not refit LDA. Screening is automated with a documented manual text sample; it is not a manual census or an audio audit.

Run cells in order. Compact saved district profiles make reruns fast. Set `REBUILD_FROM_TRANSCRIPTS=True` only to rescan the full cleaned corpus. Source data remain unchanged. The adjacent `workflow.py` contains the complete screening, aggregation, and inference implementation.

**Completed result:** 97,926 cleaned recordings screened; 83,369 pass primary eligibility. The matched comparison uses 21,878 recordings from 273 districts. HC+, min-P and sum of squares all reach the 0.0005 Monte Carlo resolution floor. The vocabulary difference includes likely substantive pandemic-related terms **and** transcription/number-format artifacts. See Sections 4–7 for robustness, test comparisons, and interpretation limits.

In [ ]:
from pathlib import Path
import json, sys, platform, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'workflow.py').exists():
    raise FileNotFoundError('Start this notebook in its containing analysis folder.')
sys.path.insert(0, str(ROOT))
import workflow
workflow.ROOT = ROOT
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
REBUILD_FROM_TRANSCRIPTS = False
RERUN_INFERENCE = False
if REBUILD_FROM_TRANSCRIPTS:
    workflow.audit()
    workflow.aggregate()
if REBUILD_FROM_TRANSCRIPTS or RERUN_INFERENCE:
    workflow.compare(B=1999)
audit = pd.read_csv(ROOT/'eligibility_audit.csv', dtype={'id':str,'leaid':str})
videos = pd.read_csv(ROOT/'analysis_videos.csv', dtype={'id':str,'leaid':str})
profiles = np.load(ROOT/'district_profiles.npz')
results = pd.read_csv(ROOT/'test_comparison.csv')
words = pd.read_csv(ROOT/'word_changes_primary.csv')
agg = json.loads((ROOT/'aggregation.json').read_text())
review_path = ROOT/'manual_review_log.csv'
review = pd.read_csv(review_path) if review_path.exists() else pd.DataFrame(columns=['id','review_phase','review_judgment','review_note','title'])
summary = json.loads((ROOT/'audit_summary.json').read_text())


## 1. Eligibility audit before hypothesis testing

The starting point is the earlier cleaning manifest: dated 2018–2023 transcripts, after missing/placeholder, short-transcript, and exact-duplicate exclusions. Earlier text cleaning removed timestamps, numerals, and non-speech tags and split transcripts into chunks. We recombine **all chunks of each video** before averaging; chunks are never treated as independent observations.

**Eligible scope:** school board/education board meetings and school-governance committees or workshops. Joint municipal–school meetings and uncertain cases are held out. A district ID alone is not evidence of eligibility: municipal meetings occur on some associated channels.

Screening order:
1. Flag possible standalone reports, updates, presentations and candidate events. These are **unresolved**, since a presentation title can also describe a full hearing.
2. Flag other governing bodies, including zoning/planning, town finance, health, elections, councils and charter review committees. Joint school–municipal titles are unresolved.
3. Include explicit school-governance titles.
4. For generic governance titles, require either a direct school-governance phrase in the opening transcript, or both a predominantly school-governance channel (≥10 explicit titles; ≥50% of its retained titles) and ≥3 education-word mentions covering ≥2 distinct terms in the first 600 words.
5. Hold remaining ambiguous cases out.

The primary panel uses steps 3–4; the strict panel uses step 3 only. Prefix evidence is limited to the first 600 words, capped at 3,500 characters for direct phrase matching. This can miss valid meetings and occasionally mistake a mention of a school board for the identity of the meeting itself. The exact rules are preserved in `workflow.py`.

In [ ]:
cleaning = pd.read_csv(workflow.LDA/'cleaning_manifest.csv', usecols=['status'])
display(cleaning.status.value_counts().rename_axis('earlier_cleaning_status').to_frame('records'))
counts = audit.audit_status.value_counts().rename_axis('screening_status').to_frame('videos')
counts['percent'] = 100*counts.videos/len(audit)
display(counts.round(2))
display(pd.crosstab(audit.year, audit.audit_status))
fig, ax = plt.subplots(figsize=(10,4))
counts.videos.sort_values().plot.barh(ax=ax, color='#376e8c')
ax.set(xlabel='Retained transcript recordings screened', ylabel='', title='Eligibility screening: all 2018–2023 retained recordings')
plt.tight_layout(); plt.show()
print(f"Primary eligible: {audit.primary_eligible.sum():,} / {len(audit):,}")


### Manual review and what changed

A first stratified sample of 120 records was read using titles and cleaned transcript openings. Thirty further rule-check records were inspected (some may overlap). This review found a town finance-board meeting discussing schools and standalone school-board report clips among initial inclusions. It also found genuine school meetings behind generic titles. Rules were revised before aggregation. A presentation flag proved imperfect: some flagged recordings are full meetings/hearings, so this category is labeled unresolved rather than a confirmed non-meeting.

The log records judgments and uncertainty, including poor automatic speech recognition (ASR). It is a **rule-development sample, not an independent accuracy estimate**. No audio/video was played; full recording completeness, meeting-to-video splitting, date accuracy, and district/channel linkage remain unaudited. Eligible-looking titles are not proof that every word is correctly transcribed. The generated `manual_review_candidates.json` is a candidate list, not a claim that every listed record received manual review; only `manual_review_log.csv` records reviewed evidence.

In [ ]:
if review.empty:
    display(Markdown('Historical manual-review evidence is not distributed in Git. Obtain it privately to reproduce this section; audit() generates candidates, not human labels.'))
else:
    print(f'{len(review)} review records; {review.id.nunique()} unique videos')
    display(pd.crosstab(review.review_phase, review.review_judgment))
    examples = review[review.id.isin(['P69RG9kmxeE','qv28YEAkzBk','mYDYNZ3KbhM','FydwB1THBFg','jVypsTv-N2Y','qLPDNLEQJjk'])]
    display(examples[['id','title','review_judgment','review_note']])
    # Full review evidence is available locally, with a source-video link for follow-up.
    review['video_url'] = 'https://www.youtube.com/watch?v=' + review.id

## 2. Comparable districts and the quantity being tested

Require **at least 4 eligible recordings and at least 3 distinct calendar quarters in each of 2018, 2019, 2022 and 2023**. Apply the rule separately under primary and strict eligibility. This improves longitudinal comparability but does not establish representative nationwide coverage or perfectly matched seasonality. A recording is the weight unit within a year; split recordings could still overweight a meeting. No new random subsampling is used.

For word $w$ in video $v$, let $q_{vw}$ be its count divided by the video's total count **within the fixed 12,000-word LDA vocabulary**. Average $q$ equally over videos within district-year, then equally over the two years in a period. For district $d$:

$$D_{dw}=\tfrac12(\bar q_{d,2022,w}+\bar q_{d,2023,w})-\tfrac12(\bar q_{d,2018,w}+\bar q_{d,2019,w}).$$

Each district has equal weight. Thus the estimand is the **mean district-level change in modeled-vocabulary share among this selected panel**, not change in pooled national token counts. Relative shares are compositional: a rise in one term can reduce other shares without their absolute counts falling.

The vocabulary was learned during earlier LDA work across 2018–2023. We keep it fixed and apply a symmetric pooled support filter: a word must occur in ≥max(30, ceil(10% of districts)) districts across the four years, have nonzero contrast variance, and not be `th`, `nd`, `rd`, or `st`. This is an exploratory follow-up, not an untouched confirmation sample. Coverage filtering changes which districts the conclusion describes.

In [ ]:
coverage = videos.groupby('year').agg(recordings=('id','size'), districts=('leaid','nunique'))
display(coverage)
print(json.dumps(agg, indent=2))
display(audit[audit.year.isin(workflow.YEARS)].groupby('year').agg(
    cleaned=('id','size'), primary_eligible=('primary_eligible','sum'), strict_eligible=('strict_eligible','sum')))
fig, ax = plt.subplots(figsize=(7,3))
coverage.recordings.plot.bar(ax=ax, color=['#467d99','#467d99','#c37844','#c37844'], rot=0)
ax.set(ylabel='Recordings', xlabel='Year', title='Primary matched panel: available eligible recordings')
plt.tight_layout(); plt.show()


## 3. Three tests of the same global null

The null is $E[D_{dw}]=0$ for every retained word. With $n$ districts, use
$$Z_w=\sqrt n\,\bar D_w/s_w,\qquad p_w=2\Phi(-|Z_w|).$$
These marginal normal p-values are approximate; they are **not** reported as discoveries with controlled false discovery rate.

- **Minimum p-value / max |Z|:** $T_{\max}=\max_w |Z_w|$. Concentrates on the strongest individual standardized change. The two statistics give the same ordering.
- **Higher Criticism (HC+):** order the $p$ marginal p-values as $p_{(i)}$ and take
$$T_{HC}=\max_{i\le p/2,\;p_{(i)}>1/p}\frac{\sqrt p\,[i/p-p_{(i)}]}{\sqrt{p_{(i)}(1-p_{(i)})}}.$$
The maximum is floored at zero. This truncated HC version aggregates an excess of small p-values while excluding the most extreme tail. It is intended to detect collections of relatively sparse signals; it is not guaranteed to beat the other tests.
- **Dense sum of squares:** $T_2=\sum_w Z_w^2$. Accumulates changes across many words. We do **not** use an independent chi-square reference distribution.

### Dependence-aware, approximate calibration

Words within a transcript are dependent, as are meetings within a district. The sampling unit is the **district contrast vector**. Set $R_{dw}=(D_{dw}-\bar D_w)/s_w$. For each replicate draw one independent $g_d\sim N(0,1)$ per district and use it for every word:
$$Z_w^*=\frac{1}{\sqrt{n-1}}\sum_d g_d R_{dw}.$$
This produces unit marginal variance and the plug-in sample correlation across words. Recompute all three statistics on each vector. With $B=1,999$ replicates, the Monte Carlo tail estimate is $(1+\#\{T^*\ge T\})/(B+1)$; its minimum reportable value is **0.0005**. That floor is not an exact tail probability.

This is a null-centered Gaussian district-multiplier approximation with fixed standard errors, **not** a token permutation, a finite-sample exact test, or a nonparametric proof. It assumes adequate district-level normal approximation and effectively independent district contrasts. Shared state/national shocks and sparse heavy-tailed word frequencies can violate these assumptions. All three global tests are reported together; sensitivity results are not independent replications.

In [ ]:
# The exact statistic implementation used for observed and bootstrap vectors.
import inspect
print(inspect.getsource(workflow.stats))
display(results[results.analysis.eq('primary')].set_index('test').round(5))
null = np.load(ROOT/'test_null_primary.npz')['statistics']
obs = results[results.analysis.eq('primary')].statistic.to_numpy()
fig, axes = plt.subplots(1,3,figsize=(13,3.5))
for j, (ax, name) in enumerate(zip(axes,['HC+','Max |Z|','Sum of squares'])):
    ax.hist(null[:,j], bins=35, color='#aac6d5')
    ax.axvline(np.quantile(null[:,j],.95), color='#ad493a', lw=2, label='Null 95th percentile')
    ax.set(title=f'{name} | observed: {obs[j]:,.1f}', xlabel='Null statistic (observed lies beyond this range)', ylabel='Bootstrap replicates')
    ax.legend()
plt.tight_layout(); plt.show()


## 4. Eligibility and transcription sensitivities

1. **Strict:** explicit school-governance titles only, with the coverage rule recalculated.
2. **Primary on common districts:** primary eligibility restricted to districts that also qualify for the strict panel. Compare this with strict to separate eligibility changes from district-composition changes.
3. **Artifact-word sensitivity:** same primary profiles and denominator, omitting `foreign`, `music`, `applause`, `inaudible`, and `laughter` from the tested coordinates. The review already showed repeated “foreign” in transcript openings. Some of these words also have legitimate uses; this is a robustness check, not a complete ASR correction. It does not renormalize shares after removal, so artifacts can still affect denominators.

A small p-value in all settings supports a global vocabulary difference under the calibration assumptions. It does not prove that HC has better power, that specific words are significant after selection, or that a political/curricular change caused the difference.

In [ ]:
display(results[['analysis','test','districts','words','statistic','bootstrap_p']].round(5))


## 5. Which vocabulary changes drive interpretation?

Below are large **standardized** positive and negative changes, together with absolute effects per 10,000 modeled-vocabulary tokens. Z-scores rank strength relative to between-district variation; they are not effect sizes. These are selected exploratory terms, without simultaneous confidence intervals or FDR control. They can include transcript artifacts, procedural language, and genuine substantive changes. A global rejection does not license labeling every ranked term a discovery.

In [ ]:
cols = ['word','early_frequency','late_frequency','change_per_10000','se_per_10000','z']
display(Markdown('**Largest positive standardized changes**'))
display(words.nlargest(15,'z')[cols].round(5))
display(Markdown('**Largest negative standardized changes**'))
display(words.nsmallest(15,'z')[cols].round(5))
selected = pd.concat([words.nsmallest(10,'z'), words.nlargest(10,'z')]).sort_values('z')
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(selected.word, selected.change_per_10000, color=np.where(selected.change_per_10000>0,'#bc7847','#447b98'))
ax.axvline(0, color='#555555', lw=1)
ax.set(xlabel='Mean district change per 10,000 modeled-vocabulary tokens', title='Selected standardized changes; bars show effect size')
plt.tight_layout(); plt.show()


## 6. Comparing test behavior beyond one observed dataset

The data contain only one observed period contrast. If every test rejects, this **does not establish which test is best**. To illustrate sensitivity to signal sparsity, generate 300 independent conditional Gaussian vectors with the estimated primary cross-word correlation, then add fixed randomly signed signals to 10, 100 or 1,000 randomly chosen coordinates. Added magnitudes are 1, 2 or 3 **Z units**, not changes in raw word frequency. For each scenario, compare rejection rates against the earlier bootstrap 95th-percentile thresholds.

These are controlled contrast-level illustrations, not raw-transcript simulations: they do not preserve compositional feasibility, simulate eligibility error, or establish power in the real population. Coordinates differ across scenarios; seeds are fixed, and the same noise draws are reused to make comparisons less noisy. With 300 replicates the binomial Monte Carlo standard error can be as high as 0.029. A null rejection rate near 5% only checks this conditional Gaussian construction, not the district independence assumption.

In [ ]:
simulation = pd.read_csv(ROOT/'gaussian_power_simulation.csv')
display(simulation.pivot(index=['signals','effect_z'],columns='test',values='rejection_rate').round(3))
fig, axes = plt.subplots(1,3,figsize=(12,3.5),sharey=True)
for ax, signals in zip(axes,[10,100,1000]):
    for name, g in simulation[simulation.signals.eq(signals)].groupby('test'):
        ax.plot(g.effect_z,g.rejection_rate, marker='o', label=name)
    ax.axhline(.05,color='gray',ls=':',lw=1)
    ax.set(title=f'{signals:,} shifted coordinates',xlabel='Added shift (Z units)',ylim=(0,1.04),xticks=[1,2,3])
axes[0].set_ylabel('Conditional Gaussian rejection rate')
axes[-1].legend()
plt.tight_layout(); plt.show()


## 7. Interpretation and boundaries

The comparison concerns **recorded, cleaned, eligible school-governance speech in districts with sustained coverage**. It does not identify causes of change, public opinion, individual speakers' beliefs, or changes in all U.S. school boards. Pandemic funding, virtual-meeting language, meeting agendas, recording practices, and ASR versions can all change vocabulary.

The audit is useful but unfinished at the level of gold-standard labels: unknown cases are conservatively held out, imperfect title/prefix rules remain, and strict-title results may favor consistently named channels. The two periods have different recording counts despite balanced district/year weighting. Dates and district mappings come from existing metadata and were not independently verified.

The next substantive step is to verify selected terms in raw transcript contexts and, where consequential, audio; then predefine thematic word groups or a held-out validation exercise. A state-level dependence sensitivity would be useful before making stronger inferential claims. A word-level localization method with appropriate multiplicity control is a separate analysis from these global tests.

Methodological context: Higher Criticism uses the sparse global-testing perspective discussed in STAT 236. This implementation tests paired district means; it is **not** the exact two-sample token-frequency-table procedure in [Higher criticism for two-sample frequency tables](https://arxiv.org/abs/2007.01958). Dependence is handled here by a plug-in Gaussian district calibration rather than an independent-word reference law.

### Findings from this run

- The primary analysis contains **21,878 recordings, 273 districts and 11,804 tested words**. The strict-title panel contains 216 districts.
- All three global tests have a Monte Carlo tail estimate of **0.0005**, the resolution floor from 1,999 bootstrap draws, in the primary analysis and all three sensitivity analyses. This is strong evidence of a vocabulary-composition difference **under the approximate calibration assumptions**. It is not evidence that the methods have equal power, and a smaller numerical statistic across different tests has no meaning.
- Large positive standardized changes include **pandemic (+3.30 per 10,000 modeled tokens), COVID (+1.88), ESSER (+1.20), shortages (+0.24), and inflation (+0.63)**. They are interpretable candidates, not causal findings.
- **Covet (+1.48)** has the largest absolute Z-score; coven, coveted, kovid and covert also rise. These may include misrecognitions of COVID. The number-word declines (eighty, seventy, thirty, ninety, thousand) could reflect caption-number formatting and the earlier removal of digit tokens, in addition to changed speech. The current five-word artifact check does not address those effects.
- In the conditional Gaussian illustration with 10 shifted words at +/−3 Z units, rejection rates are **43.7% for min-P, 3.3% for HC+, and 3.0% for sum of squares**. With 100 shifts at the same strength they are **98.7%, 98.3%, and 14.0%**. Thus this HC truncation can miss extremely sparse signals; no single test wins universally. The null-scenario rates are 6.0%, 3.3%, and 2.3%, respectively; 300 draws leave appreciable simulation uncertainty.

The strongest defensible conclusion is that recorded vocabulary differs across the periods, with a mixture of likely substantive changes and measurement effects. **Do not interpret rejection as proof of a political shift or as validation of ASR quality.**

### Transcript-context spot checks

The adjacent context file contains deterministic first matching excerpts from eligible primary-panel recordings, with up to two different videos per word and period. These are illustrative checks, not a representative sample or audio-validated corrections. Chunk numbers help locate the cleaned evidence.

In [ ]:
if not (ROOT/'word_context_examples.csv').exists():
    display(Markdown('Run collect_contexts.py locally to generate transcript-context excerpts; they are excluded from Git.'))
else:
    contexts = pd.read_csv(ROOT/'word_context_examples.csv')
    for row in contexts[contexts.word.isin(['covet','coven','principles','esser']) & contexts.period.eq('late')].itertuples():
        display(Markdown(f"**{row.word} · {row.year} · {row.title}** ([video](https://www.youtube.com/watch?v={row.id}), cleaned chunk {row.chunk})\n\n> {row.context}"))

In [ ]:
# Integrity checks tied to the design.
assert audit.id.is_unique and videos.id.is_unique
assert not audit.loc[audit.primary_eligible, 'other_body_title'].any()
assert not audit.loc[audit.primary_eligible, 'clip_or_campaign_title'].any()
assert set(videos.id) <= set(audit.loc[audit.primary_eligible,'id'])
coverage_check = videos.assign(quarter=pd.to_datetime(videos.meeting_date).dt.quarter).groupby(['leaid','year']).agg(n=('id','size'),q=('quarter','nunique'))
assert (coverage_check.n>=4).all() and (coverage_check.q>=3).all()
assert (coverage_check.reset_index().groupby('leaid').year.nunique()==4).all()
for mode in ['primary','strict']:
    for period in ['early','late']:
        x=profiles[f'{mode}_{period}']
        assert np.isfinite(x).all() and (x>=0).all()
        assert np.allclose(x.sum(axis=1),1)
assert not agg['empty_videos']
assert np.isfinite(results.statistic).all()
assert ((results.bootstrap_p>=1/2000)&(results.bootstrap_p<=1)).all()
# Verify multiplier normalization analytically, without relying on a lucky simulation.
D = profiles['primary_late']-profiles['primary_early']
sd=D.std(axis=0,ddof=1); ok=sd>1e-10
R=(D[:,ok]-D[:,ok].mean(axis=0))/sd[ok]
assert np.allclose((R*R).sum(axis=0)/(len(D)-1),1)
assert np.allclose(R.mean(axis=0),0,atol=1e-10)
print('Design and numerical integrity checks passed.')
import scipy, sklearn, pyarrow, nbformat
versions={name:module.__version__ for name,module in [('numpy',np),('pandas',pd),('scipy',scipy),('sklearn',sklearn),('pyarrow',pyarrow),('nbformat',nbformat)]}
versions['python']=platform.python_version()
print(json.dumps(versions,indent=2))
print('Seed:',workflow.SEED,'; bootstrap replicates: 1999; Gaussian illustration replicates: 300')
print('Workflow SHA256:',hashlib.sha256((ROOT/'workflow.py').read_bytes()).hexdigest())
